# Digital Signals Theory

In [1]:
%matplotlib inline

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import soundfile as sf

# Chapter 12: Analyzing IIR filters

> ...(Infinite impulse response) filters can be both more powerful and more efficient than FIR (convolutional) filters. However, these benefits come with a cost: IIR filters cannot be analyzed _directly_ by the discrete Fourier transform.

## 12.1. The z-transform

> ... can we analyze IIR filters in the frequency domain as well?
> It turns out that the answer is yes, but **not** with the discrete Fourier transform. We’ll need a new tool, which is known as the _z-transform..._

### 12.1.1. Why can’t we use the DFT?

The DFT is defined in terms of a _finite_ number of analysis frequencies: those which complete an integral number of cycles over a fixed duration of time ($N$ samples).

Consider the case of an IIR filter, such as the exponential moving average we saw earlier as

\begin{flalign}
\qquad {\color{#984ea3}y[n]} &= {\color{#e41a1c}\frac{1}{2}} \cdot {\color{#377eb8}x[n]} + {\color{#4daf4a}\frac{1}{2}} \cdot {\color{#984ea3}y[n-1]} & \text{(11.1)}  \\
\end{flalign}

Its impulse response can be infinite in length: there is no finite $N$ for which all $y[n > 0] = 0$, and so we cannot even begin to construct a DFT.

### 12.1.2. Defining the z-transform

We start with the definition of the Discrete Fourier Transform for a frequency index $m$:

\begin{flalign}
\qquad X[m] &= \sum_{n=0}^{{\color{#377eb8}N-1}} x[n] \cdot e^{-j \cdot 2\pi \cdot \frac{m}{{\color{#377eb8}N}} \cdot n} & \\
\end{flalign}

Note that ${\color{#377eb8}N}$ appears in two places: a) in the limit of the summation; and b) in the denominator in the complex exponential. This is the key limitation of the DFT that prevents us from applying it in the case of IIR, so let's see about removing this limitation.

##### _Leverage the exponent multiplication rule_

Recall that with exponents, powers can be represented as multiplication and vice-versa.

\begin{flalign}
\qquad e^{a \cdot b} &= \left( e^a \right)^b & \\
\end{flalign}

So in factoring out that $-n$ in the complex exponential bit, we can rewrite that as:

\begin{flalign}
\qquad e^{-j \cdot 2\pi \cdot \frac{m}{N} \cdot n} &= \left( e^{j \cdot 2\pi \cdot \frac{m}{N}} \right)^{-n} & \\
\end{flalign}

And now we can re-interpret the complex exponential bit as a sequence, _parameterized by sample index $n$_, and _generated by a complex number $z_{m/N}$ on the unit circle at the angle $\theta = 2\pi \cdot \frac{m}{N}$._

\begin{flalign}
\qquad z_{m/N} &= e^{j \cdot 2\pi \cdot \frac{m}{N}} & \text{(12.1)} \\
\end{flalign}


* $\frac{m}{N}$ is the fraction of the unit circle we travel in each step $n$ of this sequence. In the image below, $\theta = \frac{2\pi}{8}$. The complex number $z_{1/8} = e^{j \cdot 2pi \cdot \frac{1}{8}}$ is raised to successive powers for $n = 0, -1, -2, -3, \ldots$. The result is a clockwise movement from ${z_{1/8}}^0$ going around in steps of $\frac{2\pi}{8}$ radians.

![McFee's Digital Signals Theory, Ch. 12, Figure 12.1](bin/dstbook_fig_12.1.png)


* So, we can transform the DFT summation up to a finite $N-1$ to another summation, but this one going up to infinity!

\begin{flalign}
\qquad X[m] &= \sum_{n=0}^{{\color{#377eb8}N-1}} x[n] \cdot e^{-j \cdot 2\pi \cdot \frac{m}{{\color{#377eb8}N}} \cdot n} & \\\\
\qquad      &= \sum_{n=0}^{\infty} x[n] \cdot \left( z_{m/N} \right)^{-n} & \\\\
\end{flalign}

where we <span style="background-color:#3FF;">_assume silence after ${\color{#377eb8}N}$ samples instead of just pure, infinite repetition._</span>

##### Definition 12.1 (The z-transform)

> Given a sequence ${\color{#377eb8}x[n]}$ (of possibly infinite length), the z-transform is a function $X(z)$ over complex numbers $z$ defined as follows:
>
> \begin{flalign}
\qquad X(z) &= \sum_{n=0}^{\infty} x[n] \cdot z^{-n} & \text{(12.2)} \\
\end{flalign}

### 12.1.3. Relating the z-transform to the DFT

* Unlike the DFT, the z-transform does not produce a finite sequence of coefficients.
* Instead, it produces a function that can be evaluated for any complex number $z$.
* So the DFT can be recovered from the z-transform by evaluating $X(z)$ at carefully chosen points $z$.
* The DFT coefficients $X[m]$ can be extracted from the _z-transform_ as follows:

\begin{flalign}
\qquad X[m] &= X \left( e^{j \cdot 2\pi \cdot \frac{m}{N}} \right) & \\\\
\end{flalign}

### 12.1.4. Aside: continuous frequency and the DTFT

* The _Discrete-time Fourier Transform_ (DTF) is a special case of the _z-transform._
* The DFT uses ${\color{#377eb8}N}$ analysis frequences $0, \frac{f_s}{{\color{#377eb8}N}}, \frac{2 \cdot f_s}{{\color{#377eb8}N}}, \frac{3 \cdot f_s}{{\color{#377eb8}N}}, \ldots$
* In contrast, the DTFT is defined for **any** frequency $f$, and not just only analysis frequencies.
* The DTFT for any frequency $f$ is obtained by evaluating the _z-transform_ at a point on the unit-circle with angle $\theta = \frac{2\pi \cdot f}{f_s}$:

\begin{flalign}
\qquad DTFT(x)(f) &= X \left( e^{j \cdot 2\pi \cdot \frac{f}{f_s}} \right) & \\\\
\end{flalign}

* And so the DTFT makes the connection between _frequency_ and _angle_ precisely: frequency $f$ with sampling frequency $f_s$ is mapped to a complex number with _unit magnitude_ and _angle_ $\theta = \frac{2\pi \cdot f}{f_s}$.
* <span style="background-color:#3FF;">Aliasing frequencies map to the same angle, as:</span>

\begin{flalign}
\qquad 2\pi \cdot \frac{f + {\color{#cb181d}k \cdot f_s}}{f_s} &= 2\pi \cdot \frac{f}{f_S} + 2\pi \cdot {\color{#cb181d}k} & \\\\
\qquad                                                         &= 2\pi \cdot \frac{f}{f_S}  & \\\\
\end{flalign}

* <span style="background-color:#3FF;">The Nyquist frequency $f = \frac{f_s}{2}$ is mapped to angle $2\pi \cdot \frac{f_s/2}{f_s} = \pi \equiv -\pi$</span>
* In the DTFT, the analysis frequencies of the DFT are generalized using $f = m \cdot \frac{f_s}{N}$, which recovers the $z_{m/N}$ explained earlier.
* Furthermore, the range of frequencies $0 \leq f \leq f_s$  are mapped continuously to the range of angles $0 \leq \theta \leq 2\pi$
. 